In [2]:
import pandas as pd

cols = ['name', 'address', 'online_order', 'book_table', 'rate', 'votes',
        'location', 'rest_type', 'cuisines', 'approx_cost(for two people)',
        'listed_in(type)', 'listed_in(city)']

df = pd.read_csv("zomato.csv.zip", usecols=cols)
df.shape

(51717, 12)

In [3]:
df = df.rename(columns={
    'approx_cost(for two people)': 'cost_for_two',
    'listed_in(type)': 'listing_type',
    'listed_in(city)': 'listing_area'
})

# rate: turn "3.9/5" and "3.9 /5" into the number 3.9
df['rate'] = (df['rate']
              .astype(str)
              .str.replace('/5', '', regex=False)
              .str.strip())
df['rate'] = pd.to_numeric(df['rate'], errors='coerce')   # "NEW" and "-" become NaN

# cost: remove commas and convert to number
df['cost_for_two'] = (df['cost_for_two']
                      .astype(str)
                      .str.replace(',', '', regex=False))
df['cost_for_two'] = pd.to_numeric(df['cost_for_two'], errors='coerce')

df[['rate', 'cost_for_two']].describe()

,rate,cost_for_two
count,41665.000000,51371.000000
mean,3.700449,555.431566
std,0.440513,438.850728
min,1.800000,40.000000
25%,3.400000,300.000000
50%,3.700000,400.000000
75%,4.000000,650.000000
max,4.900000,6000.000000


In [4]:
df.isnull().sum()

address             0
name                0
online_order        0
book_table          0
rate            10052
votes               0
location           21
rest_type         227
cuisines           45
cost_for_two      346
listing_type        0
listing_area        0
dtype: int64

In [5]:
# 1. Remove exact duplicate rows
df_listings = df.drop_duplicates().copy()
print("after exact duplicates:", df_listings.shape)

# 2. One row per restaurant (same name + same address = same place)
restaurants = df_listings.drop_duplicates(subset=['name', 'address']).copy()
print("unique restaurants:", restaurants.shape)

after exact duplicates: (51632, 12)
unique restaurants: (12499, 12)


In [6]:
restaurants.isnull().sum()

address            0
name               0
online_order       0
book_table         0
rate            3183
votes              0
location           9
rest_type         63
cuisines          19
cost_for_two      59
listing_type       0
listing_area       0
dtype: int64

In [7]:
# Rows with no location or cuisine can't be analysed by area or cuisine, so drop them
restaurants = restaurants.dropna(subset=['location', 'cuisines'])

# rest_type: label unknowns rather than lose the row
restaurants['rest_type'] = restaurants['rest_type'].fillna('Unknown')

# cost_for_two: only ~59 missing. Fill with the median cost of that location
restaurants['cost_for_two'] = restaurants.groupby('location')['cost_for_two'] \
    .transform(lambda s: s.fillna(s.median()))

# rate: leave empty on purpose (your decision)
restaurants.shape

(12480, 12)

In [12]:
restaurants.to_csv("restaurants_clean.csv", index=False)
df_listings.to_csv("listings_clean.csv", index=False)

In [9]:
check = pd.read_csv("restaurants_clean.csv")
print(check.shape)
check.head()

(12480, 12)


,address,name,online_order,book_table,rate,votes,location,rest_type,cuisines,cost_for_two,listing_type,listing_area
0,"942, 21st Main Road, 2nd Stage, Banashankari, ...",Jalsa,Yes,Yes,4.1,775,Banashankari,Casual Dining,"North Indian, Mughlai, Chinese",800.0,Buffet,Banashankari
1,"2nd Floor, 80 Feet Road, Near Big Bazaar, 6th ...",Spice Elephant,Yes,No,4.1,787,Banashankari,Casual Dining,"Chinese, North Indian, Thai",800.0,Buffet,Banashankari
2,"1112, Next to KIMS Medical College, 17th Cross...",San Churro Cafe,Yes,No,3.8,918,Banashankari,"Cafe, Casual Dining","Cafe, Mexican, Italian",800.0,Buffet,Banashankari
3,"1st Floor, Annakuteera, 3rd Stage, Banashankar...",Addhuri Udupi Bhojana,No,No,3.7,88,Banashankari,Quick Bites,"South Indian, North Indian",300.0,Buffet,Banashankari
4,"10, 3rd Floor, Lakshmi Associates, Gandhi Baza...",Grand Village,No,No,3.8,166,Basavanagudi,Casual Dining,"North Indian, Rajasthani",600.0,Buffet,Banashankari


In [10]:
restaurants.nlargest(5, 'cost_for_two')[['name', 'location', 'rest_type', 'cuisines', 'cost_for_two']]

,name,location,rest_type,cuisines,cost_for_two
19139,Le Cirque Signature - The Leela Palace,Old Airport Road,Fine Dining,"French, Italian",6000.0
40266,Royal Afghan - ITC Windsor,Sankey Road,Fine Dining,"North Indian, Mughlai",5000.0
41591,Malties - Radisson Blu,Marathahalli,Lounge,"Continental, Fast Food",4500.0
37715,La Brasserie - Le Meridien,Sankey Road,Fine Dining,"North Indian, Continental, Japanese, Chinese, ...",4100.0
6104,Alba - JW Marriott Bengaluru,Lavelle Road,Fine Dining,Italian,4000.0


In [11]:
import pandas as pd
import sqlite3

restaurants = pd.read_csv("restaurants_clean.csv")

con = sqlite3.connect("bengaluru_restaurants.db")
restaurants.to_sql("restaurants", con, if_exists="replace", index=False)

def sql(query):
    return pd.read_sql(query, con)

sql("SELECT COUNT(*) AS total_restaurants, ROUND(AVG(rate),2) AS avg_rating, ROUND(AVG(cost_for_two),0) AS avg_cost FROM restaurants")

,total_restaurants,avg_rating,avg_cost
0,12480,3.63,487.0


In [13]:
import os
print(os.getcwd())

C:\Users\Welcome
